<h1 style="
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    font-size: 36px;
    color: #2c3e50;
    background-color: #ecf0f1;
    padding: 20px;
    border-radius: 12px;
    text-align: center;
    box-shadow: 0px 4px 10px rgba(0, 0, 0, 0.1);">
    NextGen Model Implementation/Run
</h1>

**Authors:** 

<ul style="line-height:1.5;">
<li>Ayman Nassar <a href="mailto:ayman.nassar@usu.edu">(ayman.nassar@usu.edu)</a></li>
<li>David Tarboton <a href="mailto:david.tarboton@usu.edu">(david.tarboton@usu.edu)</a></li>
<li>Furqan Baig <a href="fbaig@illinois.edu">(fbaig@illinois.edu)</a></li>
</ul>

**Last Updated:** 05/15/2026

**Purpose:**

This Jupyter Notebook enables the execution of the NextGen using a [**Python wrapper**](https://github.com/fbaig/ciroh_ngiab_python/tree/main). The input data, subset for the Area of Interest (AOI), is prepared using the separate notebook titled "NextGen Data Preparation" and is utilized here to drive the model simulation.

**Audience:**

Researchers who are familiar with Jupyter Notebooks, basic Python, and basic hydrologic data analysis.

**Description:**

The [**Python wrapper**](https://github.com/fbaig/ciroh_ngiab_python/tree/main) for running NextGen allows seamless model execution directly within the Jupyter environment, eliminating the need for terminal commands.

**Data Description:**

The user must provide the path to the preprocessed data generated using the "NextGen Data Preparation" Jupyter notebook. This directory should include the hydrofabric subset, meteorological forcings, and all necessary model configuration and realization files.

**Software Requirements:**

This notebook requires the following library/package versions:

> pyngiab  

<div style="
    padding: 15px 20px; 
    background-color: #e2f0fe; 
    border-left: 6px solid #3b82f6; 
    color: #1e3a8a; 
    border-radius: 4px; 
    margin-bottom: 20px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
">
    <h3 style="margin-top: 0; color: #1e3a8a; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        💡 Quick Note Before You Begin
    </h3>
    <p style="margin-bottom: 10px; font-size: 1.05em;">
        To make sure everything runs smoothly and all settings initialize correctly, <strong>please take a moment to restart the kernel before running the cells below.</strong>
    </p>
    <p style="margin: 0; font-size: 0.95em;">
        <strong>How to do this:</strong> Simply navigate to the <strong>Kernel</strong> menu at the top and select <span style="background-color: rgba(0,0,0,0.05); padding: 2px 6px; border-radius: 3px; border: 1px solid rgba(0,0,0,0.1);"><strong>“Restart Kernel and Clear Outputs of All Cells”</strong></span>. Thank you!
    </p>
</div>

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:10px 16px; border-radius:8px; margin-top:0; margin-bottom:-6px;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    1. Prepare the Python Environment
  </h3>
</div>

<p style="margin-top:0; margin-bottom:4px; padding-top:0;">
<strong>Import all the required libraries</strong>: This section imports the essential Python libraries and packages.  
In this Jupyter notebook, the <code>pyngiab.py</code> script is used to consolidate all the necessary functions for running the NextGen model.  
More information about the <code>pyngiab.py</code> package can be found 
<a href="https://github.com/fbaig/ciroh_pyngiab/tree/main"><strong>here</strong></a>.
</p>

In [74]:
# ----------------------------- Importing Required Libraries -----------------------------

import ngiab_utils
import pandas as pd
import geopandas as gpd
from pathlib import Path
from pyngiab import PyNGIAB
import matplotlib.pyplot as plt
from ngen_outputs_utils import get_flow_data_from_netcdf, process_usgs_streamflow

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:10px 16px; border-radius:8px; margin-top:0; margin-bottom:-6px;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    1.1 How many Catchments
  </h3>
</div>

<p style="margin-top:0; margin-bottom:4px; padding-top:0;">
This section executes the NextGen model using preprocessed data. The user should provide the path to the dataset prepared using the <strong>NextGen Data Preparation</strong> Jupyter notebook.
</p>


In [75]:
import os
from pathlib import Path

os.environ["HYDRA_LAUNCHER"] = "fork"     # MPICH 4.x
os.environ["HYDRA_BOOTSTRAP"] = "fork"    # older MPICH naming, set both to be safe

# Specify the HydroFabric subset ID and locate the directory where the NGIAB
# preprocessing workflow stored the forcing data, configuration files, and realization setup.
#
# This run takes about 10 minutes to run the 2 years set in the data preparation inputs for the gage used.
#
# hydrofabric_id = "gage-13340600"
hydrofabric_id = "gage-03574500"
home_dir = Path.home()


# scratch_dir = Path("/scratch/mhchowdhury")     # or wherever your scratch lives
data_dir = f'{home_dir}/ngiab_preprocess_output/{hydrofabric_id}'



In [76]:
import geopandas as gpd
gdf = gpd.read_file(f"{data_dir}/config/{hydrofabric_id}_subset.gpkg", layer="divides")
print(len(gdf), "catchments")

108 catchments


In [77]:
!cd {data_dir} && /dmod/bin/partitionGenerator config/{hydrofabric_id}_subset.gpkg config/{hydrofabric_id}_subset.gpkg /tmp/test_partitions.json 8 '' '' && head -c 300 /tmp/test_partitions.json

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
chdir: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Reading 108 features from layer divides using ID column `divide_id`
Partitioning 108 catchments into 8 partitions.
Reading 46 features from layer nexus using ID column `id`
Validating catchments...

Number of catchments is: 108
Catchment validation completed
Found 2 remotes in partition 0
Found 7 remotes in partition 1
Found 6 remotes in partition 2
Found 6 remotes in partition 3
Found 6 remotes in partition 4
Found 15 remotes in partition 5
Found 8 remotes in partition 6
Found 1 remotes in partition 7
Found 51 total remotes (average of approximately 6 remotes per partition)
{
    "partitions":[
        {"id":0,
        "cat-ids":["cat-1011979", "cat-1011982", "cat-1011983", "cat-1011972", "cat-1011970", "cat-1011971", "cat-1011969", "cat-1011976", "cat-1011973",

In [78]:
import inspect
print(inspect.signature(PyNGIAB.__init__))


(self, data_dir: str, serial_execution_mode: bool = False, venv_path: str = '/ngen/.venv/')


In [79]:
import os
fdir = f"{data_dir}/forcings"
files = os.listdir(fdir)
print(files)
# if there's one .nc with a different name:
# os.symlink(f"{fdir}/raw_gridded_data.nc", f"{fdir}/forcings.nc")

['raw_gridded_data.nc', 'forcings.nc']


In [80]:
import os
p = f"{data_dir}/forcings/forcings.nc"
print("exists:", os.path.exists(p))
print("size:", os.path.getsize(p) if os.path.exists(p) else "MISSING")

exists: True
size: 113560131


In [81]:
import os
p = f"{data_dir}/forcings/forcings.nc"
print("islink:", os.path.islink(p))
print("lexists:", os.path.lexists(p))       # True = link entry exists
print("points to:", os.readlink(p) if os.path.islink(p) else "not a link")
print("target real path:", os.path.realpath(p))
print("target exists:", os.path.exists(os.path.realpath(p)))

islink: False
lexists: True
points to: not a link
target real path: /home/mhchowdhury/ngiab_preprocess_output/gage-03574500/forcings/forcings.nc
target exists: True


In [82]:
import os, multiprocessing
os.cpu_count = lambda: 6
multiprocessing.cpu_count = lambda: 6

test_ngiab = PyNGIAB(data_dir, serial_execution_mode=False)
test_ngiab.run()

sh: 0: getcwd() failed: No such file or directory


WARN: pydantic version: Required(v1), Found(2.7.4).
WARN: numpy version: Required(1.26.4), Found(1.26.4).
Required dependencies not found in system path, looking into /ngen/.venv/


sh: 0: getcwd() failed: No such file or directory


Valid dependencies found in venv: /ngen/.venv/
*****************
forcings exists. 2 forcings files found.
config exists. 1 config files found.
Error: Directory /home/mhchowdhury/ngiab_preprocess_output/gage-03574500/outputs does not exist.


In [84]:
import os
os.chdir(data_dir)
print("now in:", os.getcwd())

# keep the fork/slurm fixes in the same cell
os.environ["HYDRA_LAUNCHER"] = "fork"
os.environ["HYDRA_BOOTSTRAP"] = "fork"
for k in list(os.environ):
    if k.startswith("SLURM"):
        del os.environ[k]

os.cpu_count = lambda: 8
import multiprocessing
multiprocessing.cpu_count = lambda: 8

test_ngiab = PyNGIAB(data_dir, serial_execution_mode=False)
test_ngiab.run()

now in: /home/mhchowdhury/ngiab_preprocess_output/gage-03574500
WARN: pydantic version: Required(v1), Found(2.7.4).
WARN: numpy version: Required(1.26.4), Found(1.26.4).
Required dependencies not found in system path, looking into /ngen/.venv/
Valid dependencies found in venv: /ngen/.venv/
*****************
forcings exists. 2 forcings files found.
config exists. 1 config files found.
Error: Directory /home/mhchowdhury/ngiab_preprocess_output/gage-03574500/outputs does not exist.


<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:10px 16px; border-radius:8px; margin-top:0; margin-bottom:-6px;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    2. NextGen Run
  </h3>
</div>

<p style="margin-top:0; margin-bottom:4px; padding-top:0;">
This section executes the NextGen model using preprocessed data. The user should provide the path to the dataset prepared using the <strong>NextGen Data Preparation</strong> Jupyter notebook.
</p>


In [83]:


# Initialize the model for serial execution
test_ngiab = PyNGIAB(data_dir, serial_execution_mode=False)

# Run the model
test_ngiab.run()

sh: 0: getcwd() failed: No such file or directory


WARN: pydantic version: Required(v1), Found(2.7.4).
WARN: numpy version: Required(1.26.4), Found(1.26.4).
Required dependencies not found in system path, looking into /ngen/.venv/


sh: 0: getcwd() failed: No such file or directory


Valid dependencies found in venv: /ngen/.venv/
*****************
forcings exists. 2 forcings files found.
config exists. 1 config files found.
Error: Directory /home/mhchowdhury/ngiab_preprocess_output/gage-03574500/outputs does not exist.


<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:10px 16px; border-radius:8px; margin-top:0; margin-bottom:-6px;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    3. Comparison Plot Between Simulated and Observed Streamflow
  </h3>
</div>

<h4 style="background-color:#e9f0ff; color:#1b2a4e; padding:10px 14px; border-left:5px solid #4b6cff; border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:18px;">
  🔹 Set Simulation Period and Input File Paths
</h4>

<p style="margin-top:10px; font-size:15px; line-height:1.6;">
Define the simulation start and end dates, the hydrofabric GeoPackage associated with the selected gage, and the T-Route NetCDF file containing routed streamflow outputs.
</p>

<p style="margin-left:18px; font-size:15px; line-height:1.6;">
<strong>Simulation Period</strong> → <code>start_date</code>, <code>end_date</code><br>
<strong>Hydrofabric File</strong> → <code>.gpkg</code> path<br>
<strong>T-Route Output</strong> → <code>.nc</code> NetCDF path
</p>

In [ ]:
# Specify start and end dates
start_date = "2017-10-01"
end_date = "2021-09-30"

# Provide the path to your GeoPackage for the HydroFabric subset
gpkg_path = (
    f"{home_dir}/ngiab_preprocess_output/{hydrofabric_id}/config/{hydrofabric_id}_subset.gpkg"
)

# Provide the path of T-route NetCDF in outputs/troute (e.g. troute_output_202010010000.nc)
troute_path = next(Path(data_dir, "outputs/troute").glob("*.nc"))

<h4 style="background-color:#e9f0ff; color:#1b2a4e; padding:10px 14px; border-left:5px solid #4b6cff; border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:18px;">
  🔹 Look Up NGEN <code>feature_id</code> from USGS Gage ID
</h4>

<p style="margin-top:10px; font-size:15px; line-height:1.6;">
Each NGIAB hydrofabric run records the routing segment (<code>feature_id</code>) associated with each USGS stream gage. 
The cell below reads this mapping from the <code>data_dir</code> directory and builds a lookup table that links:
</p>

<p style="margin-left:18px; font-size:15px; line-height:1.6;">
<strong>USGS gage ID</strong> → <strong><code>feature_id</code></strong>
</p>

In [ ]:
# Read gage ↔ segment pairs from the hydrofabric run directory
# Returns: list of (ngen_segment_id, usgs_gage_id)
ngiab_usgs_gages = ngiab_utils.get_gages_from_hydrofabric(data_dir)

# Build lookup table: USGS gage id → NGEN feature_id
ngiab_gages_df = pd.DataFrame(ngiab_usgs_gages, columns=["feature_id", "usgs_gage"])

# USGS ids as strings (for matching and NWIS downloads)
ngiab_gages_df["usgs_gage"] = ngiab_gages_df["usgs_gage"].astype(str)

# Remove "wb-" prefix → numeric routing id used in NetCDF / troute
# Example: "wb-2861391" → 2861391
ngiab_gages_df["feature_id"] = (
    ngiab_gages_df["feature_id"]
    .astype(str)
    .str.replace("wb-", "", regex=False)
    .astype(int)
)

# Crosswalk table (one row per gage in this subset)
ngiab_gages_df

<h4 style="background-color:#e9f0ff; color:#1b2a4e; padding:10px 14px; border-left:5px solid #4b6cff; border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:18px;">
  🔹 Build Hydrograph Table: Align USGS Observed and NGEN Simulated Streamflow
</h4>

<p style="font-family:'Segoe UI', sans-serif; font-size:14px; color:#333; line-height:1.6;">
  The cell below combines <strong>simulated</strong> streamflow from the troute NetCDF (at the NGEN routing feature linked to your USGS gage) with <strong>observed</strong> USGS streamflow for the same period. Basin drainage area from the hydrofabric geopackage is used to put simulated flow in the same units as the observations (m/hr per m²). The result is a single table, <code>hydrograph_df</code>, with hourly time, simulated, and observed columns—ready for the hydrograph plot in the next cell.
</p>

In [ ]:
# Step 1: IDs from the hydrofabric lookup table
usgs_gage_id = str(ngiab_gages_df["usgs_gage"].iloc[0])           # e.g. 10109001
ngen_routing_feature_id = int(ngiab_gages_df["feature_id"].iloc[0])  # e.g. 2861391

# Step 2: Total basin drainage area (km² → m²)
# Total drainage_area in sqkm = full watershed above the outlet gage
watershed_area_m2 = (
    gpd.read_file(gpkg_path, layer="divides")["tot_drainage_areasqkm"].max() * 1e6
)

# Step 3: Simulated streamflow from troute NetCDF (m³/h)
simulated_flow_m3_per_hour = get_flow_data_from_netcdf(troute_path, ngen_routing_feature_id)

# Step 4: Observed streamflow from USGS
observed_streamflow_df = process_usgs_streamflow(usgs_gage_id, start_date, end_date, gpkg_path)

if "Streamflow (m/hr)" in observed_streamflow_df.columns:
    observed_flow_m_per_hour_m2 = observed_streamflow_df["Streamflow (m/hr)"]
else:
    observed_flow_m_per_hour_m2 = (
        observed_streamflow_df["00060"] * 3600 / (35.3147 * watershed_area_m2)
    )

# Step 5: Shortest record length (sim vs obs)
number_of_timesteps = min(len(simulated_flow_m3_per_hour), len(observed_flow_m_per_hour_m2))

# Step 6: One table: time, simulated, observed
hydrograph_df = pd.DataFrame({
    "Time": pd.to_datetime(observed_streamflow_df["Time"].iloc[:number_of_timesteps], utc=True)
              .dt.tz_convert(None).dt.floor("h"),
    "simulated_streamflow": simulated_flow_m3_per_hour[:number_of_timesteps] / watershed_area_m2,
    "observed_streamflow": observed_flow_m_per_hour_m2.iloc[:number_of_timesteps].values,
}).dropna()

print(f"USGS gage: {usgs_gage_id} | NGEN feature: {ngen_routing_feature_id} | rows: {len(hydrograph_df)}")
hydrograph_df.head()

<h4 style="background-color:#e9f0ff; color:#1b2a4e; padding:10px 14px; border-left:5px solid #4b6cff; border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:18px;">
  🔹 Plot Hydrograph: USGS Observed vs NGEN Simulated Streamflow
</h4>

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(
    hydrograph_df["Time"],
    hydrograph_df["observed_streamflow"],
    label=f"USGS {usgs_gage_id}",
)
plt.plot(
    hydrograph_df["Time"],
    hydrograph_df["simulated_streamflow"],
    label="NGEN simulated",
)

plt.ylabel("Streamflow (m/hr per m² basin)")
plt.xlabel("Time")
plt.title(f"Hydrograph — USGS gage {usgs_gage_id}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()